# Notebook 2 — Exploratory Data Analysis & Visualization
## NFL Player Performance Analysis (QB · RB · WR · TE)

**Authors:** Milan Jovkić, Uroš Petrašković  
**Course:** Analiza i Obrada Podataka  
**Date:** 2025

---

This notebook contains **dozens of graphs and analyses** across all four positions. We explore trends, distributions, correlations, and key insights that will inform our predictive models in Notebook 3.

### Table of Contents
1. Setup & Data Loading
2. **QB Analysis** — Passing trends, efficiency, win correlation, era comparison, stat padding
3. **QB Elo Ratings** — Elo distribution, top-rated QBs, career trajectories
4. **RB Analysis** — Rushing/receiving balance, workload, efficiency
5. **WR Analysis** — Target share, advanced metrics, situational splits
6. **TE Analysis** — Receiving trends, position evolution
7. **Cross-Position Comparisons** — Scoring, efficiency, era trends
8. **Feature Importance Preview** — Which stats predict performance?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 100,
    'figure.facecolor': 'white',
})

COLORS = {'QB': '#E31837', 'RB': '#003DA5', 'WR': '#FFC20E', 'TE': '#00843D'}

print("Libraries loaded. Matplotlib backend:", plt.get_backend())

In [ ]:
# Load all datasets
qb = pd.read_csv('data/fully combined/qb_master.csv')
rb = pd.read_csv('data/fully combined/rb_master.csv')
te = pd.read_csv('data/fully combined/te_master.csv')
wr = pd.read_csv('data/fully combined/wr_all_seasons.csv')
elo_career = pd.read_csv('data/nfl elo data/qb_rankings_career.csv')
elo_2025 = pd.read_csv('data/nfl elo data/qb_rankings_2025.csv')

# Parse QB wins from QBrec
def parse_wins(rec):
    try:
        parts = str(rec).split('-')
        return int(parts[0])
    except:
        return np.nan

def parse_losses(rec):
    try:
        parts = str(rec).split('-')
        return int(parts[1])
    except:
        return np.nan

qb['Wins'] = qb['QBrec'].apply(parse_wins)
qb['Losses'] = qb['QBrec'].apply(parse_losses)
qb['Win%'] = qb['Wins'] / (qb['Wins'] + qb['Losses'])
qb['Total_Games_Started'] = qb['Wins'] + qb['Losses']

# Filter starters for cleaner analysis
qb_starters = qb[qb['GS'] >= 8].copy()

# WR starters: at least 30 targets
wr_starters = wr[wr['targets'] >= 30].copy()

# RB starters: at least 50 rush attempts
rb_starters = rb[rb['Rush_Att'] >= 50].copy()

# TE starters: at least 20 targets
te_starters = te[te['Tgt'] >= 20].copy()

print(f"QB:  {len(qb)} total → {len(qb_starters)} starters (GS ≥ 8)")
print(f"RB:  {len(rb)} total → {len(rb_starters)} starters (Rush_Att ≥ 50)")
print(f"WR:  {len(wr)} total → {len(wr_starters)} starters (Targets ≥ 30)")
print(f"TE:  {len(te)} total → {len(te_starters)} starters (Targets ≥ 20)")

---
# 2. Quarterback (QB) Analysis

The quarterback position is the most important in football. We'll explore passing trends over nearly 50 years, efficiency metrics, and the relationship between stats and winning.

---

In [ ]:
# 2.1 — Distribution of Passing Yards (starters only)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(qb_starters['Yds'].dropna(), bins=40, color=COLORS['QB'], alpha=0.7, edgecolor='white')
axes[0].axvline(qb_starters['Yds'].mean(), color='black', ls='--', lw=2, label=f"Mean: {qb_starters['Yds'].mean():.0f}")
axes[0].axvline(qb_starters['Yds'].median(), color='navy', ls=':', lw=2, label=f"Median: {qb_starters['Yds'].median():.0f}")
axes[0].set_title('Distribution of Season Passing Yards')
axes[0].set_xlabel('Passing Yards')
axes[0].legend()

# Box plot by era
qb_starters['Era'] = pd.cut(qb_starters['Season'], bins=[1978,1990,2000,2010,2020,2026],
                              labels=['1979-90','1991-00','2001-10','2011-20','2021-25'])
sns.boxplot(data=qb_starters, x='Era', y='Yds', ax=axes[1], palette='Reds')
axes[1].set_title('Passing Yards by Era')
axes[1].set_ylabel('Passing Yards')

# KDE of passer rating
for era in ['1991-00', '2001-10', '2011-20', '2021-25']:
    subset = qb_starters[qb_starters['Era'] == era]['Rate'].dropna()
    if len(subset) > 5:
        subset.plot.kde(ax=axes[2], label=era, lw=2)
axes[2].set_title('Passer Rating Distribution by Era')
axes[2].set_xlabel('Passer Rating')
axes[2].legend()
axes[2].set_xlim(40, 140)

plt.tight_layout()
plt.show()

In [ ]:
# 2.2 — Passing Stats Trends Over Time
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

yearly = qb_starters.groupby('Season').agg({
    'Yds': 'mean', 'TD': 'mean', 'Rate': 'mean', 'Cmp%': 'mean'
}).reset_index()

for ax, col, title, color in zip(
    axes.flat,
    ['Yds', 'TD', 'Rate', 'Cmp%'],
    ['Avg Passing Yards', 'Avg Passing TDs', 'Avg Passer Rating', 'Avg Completion %'],
    ['#E31837', '#003DA5', '#00843D', '#FFC20E']
):
    ax.plot(yearly['Season'], yearly[col], color=color, lw=2, marker='o', ms=3)
    z = np.polyfit(yearly['Season'].dropna(), yearly[col].dropna(), 2)
    p = np.poly1d(z)
    ax.plot(yearly['Season'], p(yearly['Season']), '--', color='gray', lw=1.5, alpha=0.7)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Season')

plt.suptitle('NFL Passing Revolution: Key Metrics Over Time', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("Conclusion: All four metrics show clear upward trends, reflecting the NFL's evolution into a passing-dominant league.")

In [ ]:
# 2.3 — Win Percentage vs Key QB Stats
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

win_stats = [('Rate', 'Passer Rating'), ('Yds', 'Passing Yards'), ('TD', 'Passing TDs'),
             ('Int', 'Interceptions'), ('Cmp%', 'Completion %'), ('Y/A', 'Yards per Attempt')]

valid = qb_starters.dropna(subset=['Win%'])

for ax, (col, title) in zip(axes.flat, win_stats):
    subset = valid.dropna(subset=[col])
    ax.scatter(subset[col], subset['Win%'], alpha=0.3, s=15, color=COLORS['QB'])
    # Trend line
    mask = subset[col].notna() & subset['Win%'].notna()
    if mask.sum() > 10:
        r, p_val = stats.pearsonr(subset.loc[mask, col], subset.loc[mask, 'Win%'])
        z = np.polyfit(subset.loc[mask, col], subset.loc[mask, 'Win%'], 1)
        poly = np.poly1d(z)
        x_range = np.linspace(subset[col].min(), subset[col].max(), 100)
        ax.plot(x_range, poly(x_range), 'k--', lw=2)
        ax.set_title(f'{title}\nr = {r:.3f} (p = {p_val:.2e})', fontsize=11)
    else:
        ax.set_title(title)
    ax.set_ylabel('Win %')
    ax.set_xlabel(col)

plt.suptitle('QB Win Percentage vs Key Statistics', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("Conclusion: Passer Rating, Yards/Attempt, and Completion % show the strongest positive correlations with winning.")
print("Interceptions show a negative correlation, confirming turnovers hurt winning chances.")

In [ ]:
# 2.4 — Career Passing Yards: Top 15 QBs
career = qb.groupby('Player').agg({
    'Yds': 'sum', 'TD': 'sum', 'Int': 'sum', 'GS': 'sum', 'Season': 'count'
}).reset_index().sort_values('Yds', ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Bar chart
colors_bar = plt.cm.Reds(np.linspace(0.3, 0.9, 15))[::-1]
bars = axes[0].barh(career['Player'][::-1], career['Yds'][::-1], color=colors_bar)
axes[0].set_xlabel('Career Passing Yards')
axes[0].set_title('Top 15 QBs by Career Passing Yards', fontsize=14, fontweight='bold')
for bar, val in zip(bars, career['Yds'][::-1]):
    axes[0].text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
                 f'{val:,.0f}', va='center', fontsize=9)

# TD vs INT scatter
career_all = qb.groupby('Player').agg({'TD': 'sum', 'Int': 'sum', 'Yds': 'sum'}).reset_index()
axes[1].scatter(career_all['TD'], career_all['Int'], 
                s=career_all['Yds']/500, alpha=0.5, color=COLORS['QB'], edgecolors='white')
# Label top QBs
for _, row in career_all.nlargest(10, 'Yds').iterrows():
    axes[1].annotate(row['Player'].split()[-1], (row['TD'], row['Int']),
                     fontsize=8, ha='center', va='bottom')
axes[1].set_xlabel('Career TDs')
axes[1].set_ylabel('Career INTs')
axes[1].set_title('Career TDs vs INTs (bubble size = yards)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 2.5 — QB Correlation Heatmap (Key Passing Stats)
corr_cols = ['Yds', 'TD', 'Int', 'Cmp%', 'Y/A', 'Rate', 'QBR', 'Sk', 
             'Win%', 'AV', 'NY/A', 'ANY/A']
corr_cols = [c for c in corr_cols if c in qb_starters.columns]
corr_matrix = qb_starters[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('QB Stats Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Key correlations with Win%:", 
      corr_matrix['Win%'].drop('Win%').sort_values(ascending=False).head(5).to_string())

In [ ]:
# 2.6 — TD% vs INT% (Efficiency Quadrant)
fig, ax = plt.subplots(figsize=(12, 8))

valid = qb_starters.dropna(subset=['TD%', 'Int%'])
scatter = ax.scatter(valid['TD%'], valid['Int%'], c=valid['Season'],
                     cmap='RdYlGn', s=30, alpha=0.6, edgecolors='gray', linewidth=0.5)
plt.colorbar(scatter, label='Season')

# Quadrant lines at medians
med_td = valid['TD%'].median()
med_int = valid['Int%'].median()
ax.axhline(med_int, color='gray', ls='--', alpha=0.5)
ax.axvline(med_td, color='gray', ls='--', alpha=0.5)

# Label quadrants
ax.text(valid['TD%'].max()*0.85, valid['Int%'].min()*1.3, 'ELITE\n(High TD%, Low INT%)',
        fontsize=10, ha='center', color='green', fontweight='bold')
ax.text(valid['TD%'].min()*1.3, valid['Int%'].max()*0.85, 'RISKY\n(Low TD%, High INT%)',
        fontsize=10, ha='center', color='red', fontweight='bold')

# Highlight modern stars
stars = ['Patrick Mahomes', 'Tom Brady', 'Aaron Rodgers', 'Peyton Manning', 'Drew Brees']
for _, row in valid[valid['Player'].isin(stars)].groupby('Player')[['TD%','Int%','Season']].mean().iterrows():
    ax.annotate(_.split()[-1], (row['TD%'], row['Int%']), fontsize=9, fontweight='bold')

ax.set_xlabel('TD%', fontsize=12)
ax.set_ylabel('INT%', fontsize=12)
ax.set_title('QB Efficiency Quadrant: TD% vs INT%', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Conclusion: The best QBs cluster in the bottom-right (high TD%, low INT%). Modern QBs trend toward better efficiency.")

In [ ]:
# 2.7 — Advanced Passing Metrics
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Air Yards vs YAC
ax = axes[0, 0]
valid_adv = qb_starters.dropna(subset=['adv_pass_pass_air_yds', 'adv_pass_pass_yac'])
ax.scatter(valid_adv['adv_pass_pass_air_yds'], valid_adv['adv_pass_pass_yac'],
           alpha=0.4, s=20, color=COLORS['QB'])
ax.set_xlabel('Air Yards'); ax.set_ylabel('Yards After Catch')
ax.set_title('Air Yards vs YAC', fontweight='bold')

# Pocket Time distribution
ax = axes[0, 1]
pocket = qb_starters['adv_pass_pocket_time'].dropna()
if len(pocket) > 0:
    ax.hist(pocket, bins=30, color=COLORS['QB'], alpha=0.7, edgecolor='white')
    ax.axvline(pocket.mean(), color='black', ls='--', label=f'Mean: {pocket.mean():.2f}s')
    ax.set_title('Pocket Time Distribution', fontweight='bold')
    ax.set_xlabel('Pocket Time (seconds)')
    ax.legend()

# Pressure rate vs passer rating
ax = axes[1, 0]
valid_p = qb_starters.dropna(subset=['adv_pass_pass_pressured_pct', 'Rate'])
if len(valid_p) > 0:
    ax.scatter(valid_p['adv_pass_pass_pressured_pct'], valid_p['Rate'], alpha=0.4, s=20, color='purple')
    r, _ = stats.pearsonr(valid_p['adv_pass_pass_pressured_pct'], valid_p['Rate'])
    ax.set_title(f'Pressure Rate vs Passer Rating (r={r:.3f})', fontweight='bold')
    ax.set_xlabel('Pressured %')
    ax.set_ylabel('Passer Rating')

# On-target % vs passer rating
ax = axes[1, 1]
valid_ot = qb_starters.dropna(subset=['adv_pass_pass_on_target_pct', 'Rate'])
if len(valid_ot) > 0:
    ax.scatter(valid_ot['adv_pass_pass_on_target_pct'], valid_ot['Rate'], alpha=0.4, s=20, color='teal')
    r, _ = stats.pearsonr(valid_ot['adv_pass_pass_on_target_pct'], valid_ot['Rate'])
    ax.set_title(f'On-Target % vs Passer Rating (r={r:.3f})', fontweight='bold')
    ax.set_xlabel('On-Target %')
    ax.set_ylabel('Passer Rating')

plt.suptitle('Advanced QB Passing Metrics', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 2.8 — Stat Padding Analysis: Do Losing QBs Inflate Yards?
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

valid = qb_starters.dropna(subset=['Win%', 'Yds', 'GS'])
valid['Yds_per_game'] = valid['Yds'] / valid['GS']

# Yards per game vs Win%
ax = axes[0]
ax.scatter(valid['Win%'], valid['Yds_per_game'], alpha=0.3, s=20, color=COLORS['QB'])
# Bin by win% and show trend
bins = pd.cut(valid['Win%'], bins=8)
binned = valid.groupby(bins, observed=True)['Yds_per_game'].mean()
bin_centers = [(b.left + b.right)/2 for b in binned.index]
ax.plot(bin_centers, binned.values, 'ko-', lw=2, ms=8, label='Binned Mean')
ax.set_xlabel('Win %')
ax.set_ylabel('Passing Yards per Game')
ax.set_title('Yards/Game vs Win%: Do Losing QBs Pad Stats?', fontweight='bold')
ax.legend()

# INT rate by Win% bins
ax = axes[1]
valid_int = qb_starters.dropna(subset=['Win%', 'Int%'])
valid_int['Win_Bin'] = pd.qcut(valid_int['Win%'], q=5, labels=['Bottom 20%','20-40%','40-60%','60-80%','Top 20%'])
sns.boxplot(data=valid_int, x='Win_Bin', y='Int%', palette='RdYlGn', ax=ax)
ax.set_title('Interception Rate by Win% Tier', fontweight='bold')
ax.set_xlabel('Win % Tier')

plt.tight_layout()
plt.show()
print("Conclusion: Moderate win% QBs tend to have higher yards/game — losing teams do throw more.")
print("Top winners have the lowest INT rates, confirming ball security as key to winning.")

In [ ]:
# 2.9 — Dual Threat QBs: Passing Yards vs Rushing Yards
fig, ax = plt.subplots(figsize=(14, 8))

valid_dt = qb_starters.dropna(subset=['Yds', 'rr_Rush_Yds'])
scatter = ax.scatter(valid_dt['Yds'], valid_dt['rr_Rush_Yds'], c=valid_dt['Season'],
                     cmap='viridis', s=30, alpha=0.6, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter, label='Season')

# Highlight top rushing QBs
top_rush = valid_dt.nlargest(15, 'rr_Rush_Yds')
for _, row in top_rush.iterrows():
    ax.annotate(f"{row['Player'].split()[-1]} '{str(int(row['Season']))[-2:]}",
                (row['Yds'], row['rr_Rush_Yds']), fontsize=7, alpha=0.8)

ax.set_xlabel('Passing Yards', fontsize=12)
ax.set_ylabel('Rushing Yards', fontsize=12)
ax.set_title('Dual Threat QBs: Passing vs Rushing Yards per Season', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("The modern NFL has seen a surge in dual-threat QBs (Lamar Jackson, Josh Allen, Jalen Hurts).")

In [ ]:
# 2.10 — Play Action vs Non-Play Action Performance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pa_cols = ['adv_pass_pass_play_action', 'adv_pass_pass_play_action_pass_yds', 'Att', 'Yds']
valid_pa = qb_starters.dropna(subset=['adv_pass_pass_play_action', 'adv_pass_pass_play_action_pass_yds'])

if len(valid_pa) > 20:
    valid_pa['PA_pct'] = valid_pa['adv_pass_pass_play_action'] / valid_pa['Att'] * 100
    valid_pa['PA_yds_pct'] = valid_pa['adv_pass_pass_play_action_pass_yds'] / valid_pa['Yds'] * 100

    ax = axes[0]
    ax.scatter(valid_pa['PA_pct'], valid_pa['Rate'], alpha=0.4, s=20, color='darkorange')
    r, _ = stats.pearsonr(valid_pa['PA_pct'].dropna(), valid_pa['Rate'].dropna())
    ax.set_xlabel('Play Action %')
    ax.set_ylabel('Passer Rating')
    ax.set_title(f'Play Action Usage vs Passer Rating (r={r:.3f})', fontweight='bold')

    ax = axes[1]
    yearly_pa = valid_pa.groupby('Season')['PA_pct'].mean()
    ax.plot(yearly_pa.index, yearly_pa.values, 'o-', color='darkorange', lw=2)
    ax.set_xlabel('Season')
    ax.set_ylabel('Avg Play Action %')
    ax.set_title('Play Action Usage Over Time', fontweight='bold')
else:
    axes[0].text(0.5, 0.5, 'Insufficient play action data', ha='center', va='center', transform=axes[0].transAxes)
    axes[1].text(0.5, 0.5, 'Insufficient play action data', ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# 2.11 — Greatest QB Seasons by Passer Rating
top_seasons = qb_starters.nlargest(20, 'Rate')[['Player','Season','Team','Yds','TD','Int','Rate','Win%']].copy()
top_seasons['Label'] = top_seasons['Player'] + ' (' + top_seasons['Season'].astype(int).astype(str) + ')'

fig, ax = plt.subplots(figsize=(14, 8))
colors_top = plt.cm.RdYlGn(np.linspace(0.3, 0.95, 20))
bars = ax.barh(top_seasons['Label'][::-1], top_seasons['Rate'][::-1], color=colors_top)
for bar, row in zip(bars, top_seasons.iloc[::-1].itertuples()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{row.Rate:.1f} ({row.TD}TD/{row.Int}INT)', va='center', fontsize=9)
ax.set_xlabel('Passer Rating')
ax.set_title('Top 20 QB Seasons by Passer Rating', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 3. QB Elo Ratings Analysis

The Elo rating system provides a composite measure of QB performance, similar to chess ratings. Higher Elo = better performance relative to peers.

---

In [ ]:
# 3.1 — Elo Distribution & Top QBs
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Distribution
ax = axes[0]
ax.hist(elo_career['QB Elo'].dropna(), bins=40, color=COLORS['QB'], alpha=0.7, edgecolor='white')
ax.axvline(elo_career['QB Elo'].mean(), color='black', ls='--', label=f"Mean: {elo_career['QB Elo'].mean():.0f}")
ax.set_title('QB Elo Rating Distribution')
ax.set_xlabel('QB Elo')
ax.legend()

# Top career Elo
top_elo = elo_career.groupby('QB')['QB Elo'].max().sort_values(ascending=False).head(15)
ax = axes[1]
bars = ax.barh(top_elo.index[::-1], top_elo.values[::-1], color=plt.cm.Reds(np.linspace(0.3,0.9,15)))
ax.set_title('Top 15 Peak Career Elo')
ax.set_xlabel('QB Elo')

# 2025 top QBs
top_2025 = elo_2025.nlargest(15, 'QB Elo')[['QB', 'QB Elo']]
ax = axes[2]
bars = ax.barh(top_2025['QB'][::-1], top_2025['QB Elo'][::-1], color=plt.cm.Blues(np.linspace(0.3,0.9,15)))
ax.set_title('Top 15 QBs — 2025 Elo')
ax.set_xlabel('QB Elo')

plt.tight_layout()
plt.show()

In [ ]:
# 3.2 — Elo Career Trajectories (Top QBs)
fig, ax = plt.subplots(figsize=(14, 8))

top_qbs = elo_career.groupby('QB')['QB Elo'].max().nlargest(8).index
for qb_name in top_qbs:
    player_data = elo_career[elo_career['QB'] == qb_name].sort_values('Season')
    ax.plot(player_data['Season'], player_data['QB Elo'], 'o-', label=qb_name, lw=2, ms=5)

ax.set_xlabel('Season')
ax.set_ylabel('QB Elo')
ax.set_title('Career Elo Trajectories — Top 8 QBs', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()
print("We can see peaks, declines, and the emergence of new talent across seasons.")

In [ ]:
# 3.3 — Wins vs Elo Rating
fig, ax = plt.subplots(figsize=(12, 7))

elo_wins = elo_career.dropna(subset=['QB Elo', 'W'])
ax.scatter(elo_wins['QB Elo'], elo_wins['W'], alpha=0.3, s=20, color=COLORS['QB'])
r, p = stats.pearsonr(elo_wins['QB Elo'], elo_wins['W'])
z = np.polyfit(elo_wins['QB Elo'], elo_wins['W'], 1)
poly = np.poly1d(z)
x_line = np.linspace(elo_wins['QB Elo'].min(), elo_wins['QB Elo'].max(), 100)
ax.plot(x_line, poly(x_line), 'k--', lw=2)
ax.set_xlabel('QB Elo Rating')
ax.set_ylabel('Season Wins')
ax.set_title(f'QB Elo vs Season Wins (r = {r:.3f}, p < 0.001)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Correlation: r = {r:.3f} — Higher Elo strongly correlates with more wins.")

---
# 4. Running Back (RB) Analysis

Running backs are the workhorse of the offense. We'll analyze rushing efficiency, the pass-catching evolution, workload effects, and career arcs.

---

In [ ]:
# 4.1 — RB Rushing Yards Distribution & Trends
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution
ax = axes[0]
ax.hist(rb_starters['Rush_Yds'].dropna(), bins=35, color=COLORS['RB'], alpha=0.7, edgecolor='white')
ax.axvline(rb_starters['Rush_Yds'].mean(), color='black', ls='--', label=f"Mean: {rb_starters['Rush_Yds'].mean():.0f}")
ax.set_title('Distribution of Season Rushing Yards')
ax.set_xlabel('Rushing Yards')
ax.legend()

# Trend over time
ax = axes[1]
yearly_rb = rb_starters.groupby('Season').agg({'Rush_Yds': 'mean', 'Rush_Y/A': 'mean'}).reset_index()
ax.plot(yearly_rb['Season'], yearly_rb['Rush_Yds'], 'o-', color=COLORS['RB'], lw=2)
ax.set_title('Avg Rushing Yards Per Season Over Time')
ax.set_xlabel('Season')
ax.set_ylabel('Avg Rushing Yards')

# Rushing Y/A trend
ax = axes[2]
ax.plot(yearly_rb['Season'], yearly_rb['Rush_Y/A'], 's-', color='darkblue', lw=2)
ax.set_title('Avg Yards Per Attempt Over Time')
ax.set_xlabel('Season')
ax.set_ylabel('Y/A')

plt.tight_layout()
plt.show()

In [ ]:
# 4.2 — Top 15 Career Rushing Yards
rb_career = rb.groupby('Player').agg({
    'Rush_Yds': 'sum', 'Rush_TD': 'sum', 'Rush_Att': 'sum', 'Rec_Yds': 'sum',
    'GS': 'sum', 'Season': 'count'
}).reset_index()
rb_career['Rush_Y/A_career'] = rb_career['Rush_Yds'] / rb_career['Rush_Att']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top15 = rb_career.nlargest(15, 'Rush_Yds')
colors_bar = plt.cm.Blues(np.linspace(0.3, 0.9, 15))[::-1]
bars = axes[0].barh(top15['Player'][::-1], top15['Rush_Yds'][::-1], color=colors_bar)
for bar, val in zip(bars, top15['Rush_Yds'][::-1]):
    axes[0].text(bar.get_width()+200, bar.get_y()+bar.get_height()/2, f'{val:,.0f}', va='center', fontsize=9)
axes[0].set_xlabel('Career Rushing Yards')
axes[0].set_title('Top 15 RBs by Career Rushing Yards', fontweight='bold')

# Rushing TDs
top15_td = rb_career.nlargest(15, 'Rush_TD')
bars = axes[1].barh(top15_td['Player'][::-1], top15_td['Rush_TD'][::-1],
                     color=plt.cm.Greens(np.linspace(0.3, 0.9, 15)))
axes[1].set_xlabel('Career Rushing TDs')
axes[1].set_title('Top 15 RBs by Career Rushing TDs', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 4.3 — RB Dual Threat: Rushing vs Receiving
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter
ax = axes[0]
valid_rb = rb_starters.dropna(subset=['Rush_Yds', 'Rec_Yds'])
scatter = ax.scatter(valid_rb['Rush_Yds'], valid_rb['Rec_Yds'], c=valid_rb['Season'],
                     cmap='viridis', s=25, alpha=0.5, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter, ax=ax, label='Season')
ax.set_xlabel('Rushing Yards')
ax.set_ylabel('Receiving Yards')
ax.set_title('RB Dual Threat: Rushing vs Receiving', fontweight='bold')

# Receiving share over time (what % of RB yards come from receiving?)
ax = axes[1]
rb_starters_valid = rb_starters.dropna(subset=['Rush_Yds', 'Rec_Yds']).copy()
rb_starters_valid['Rec_Share'] = rb_starters_valid['Rec_Yds'] / (rb_starters_valid['Rush_Yds'] + rb_starters_valid['Rec_Yds']) * 100
yearly_rec = rb_starters_valid.groupby('Season')['Rec_Share'].mean()
ax.plot(yearly_rec.index, yearly_rec.values, 'o-', color=COLORS['RB'], lw=2)
ax.fill_between(yearly_rec.index, yearly_rec.values, alpha=0.2, color=COLORS['RB'])
ax.set_xlabel('Season')
ax.set_ylabel('Receiving Yards Share (%)')
ax.set_title('RB Receiving Share Over Time', fontweight='bold')

plt.tight_layout()
plt.show()
print("Conclusion: Modern RBs catch more passes than ever. The receiving share has steadily increased.")

In [ ]:
# 4.4 — RB Workload vs Efficiency
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Attempts vs Y/A
ax = axes[0]
valid_rb_eff = rb_starters.dropna(subset=['Rush_Att', 'Rush_Y/A'])
ax.scatter(valid_rb_eff['Rush_Att'], valid_rb_eff['Rush_Y/A'], alpha=0.3, s=20, color=COLORS['RB'])
r, _ = stats.pearsonr(valid_rb_eff['Rush_Att'], valid_rb_eff['Rush_Y/A'])
ax.set_xlabel('Rush Attempts')
ax.set_ylabel('Yards Per Attempt')
ax.set_title(f'Workload vs Efficiency (r={r:.3f})', fontweight='bold')

# Age curve
ax = axes[1]
age_stats = rb_starters.groupby('Age').agg({'Rush_Y/A': 'mean', 'Rush_Yds': 'mean'}).reset_index()
age_stats = age_stats[(age_stats['Age'] >= 20) & (age_stats['Age'] <= 35)]
ax.plot(age_stats['Age'], age_stats['Rush_Y/A'], 'o-', color=COLORS['RB'], lw=2, label='Y/A')
ax2 = ax.twinx()
ax2.bar(age_stats['Age'], age_stats['Rush_Yds'], alpha=0.2, color=COLORS['RB'], label='Avg Rush Yds')
ax.set_xlabel('Age')
ax.set_ylabel('Yards Per Attempt', color=COLORS['RB'])
ax2.set_ylabel('Avg Rushing Yards', alpha=0.5)
ax.set_title('RB Age Curve: Efficiency & Production', fontweight='bold')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()
print("Running backs typically peak at age 25–27 and decline rapidly after 28.")

In [ ]:
# 4.5 — RB Correlation Heatmap
rb_corr_cols = ['Rush_Att', 'Rush_Yds', 'Rush_TD', 'Rush_Y/A', 'Tgt', 'Rec', 
                'Rec_Yds', 'Rec_TD', 'Scrimmage_Yds', 'Touches']
rb_corr_cols = [c for c in rb_corr_cols if c in rb_starters.columns]
corr = rb_starters[rb_corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Blues', vmin=-0.5, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('RB Stats Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 4.6 — Advanced RB Metrics: Yards Before/After Contact
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

adv_rb_cols = ['adv_Rush_YBC', 'adv_Rush_YAC', 'adv_Rush_BrkTkl']
valid_adv_rb = rb_starters.dropna(subset=['adv_Rush_YBC', 'adv_Rush_YAC'])

if len(valid_adv_rb) > 10:
    # YBC vs YAC
    ax = axes[0]
    ax.scatter(valid_adv_rb['adv_Rush_YBC'], valid_adv_rb['adv_Rush_YAC'],
               alpha=0.4, s=25, color=COLORS['RB'])
    ax.set_xlabel('Yards Before Contact')
    ax.set_ylabel('Yards After Contact')
    ax.set_title('RB Yards Before Contact vs After Contact', fontweight='bold')

    # Broken tackles vs Rush Yds
    ax = axes[1]
    valid_bt = rb_starters.dropna(subset=['adv_Rush_BrkTkl', 'Rush_Yds'])
    if len(valid_bt) > 10:
        ax.scatter(valid_bt['adv_Rush_BrkTkl'], valid_bt['Rush_Yds'],
                   alpha=0.4, s=25, color='darkblue')
        r, _ = stats.pearsonr(valid_bt['adv_Rush_BrkTkl'], valid_bt['Rush_Yds'])
        ax.set_xlabel('Broken Tackles')
        ax.set_ylabel('Rushing Yards')
        ax.set_title(f'Broken Tackles vs Rushing Yards (r={r:.3f})', fontweight='bold')
else:
    axes[0].text(0.5, 0.5, 'Insufficient advanced data', ha='center', va='center', transform=axes[0].transAxes)
    axes[1].text(0.5, 0.5, 'Insufficient advanced data', ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

---
# 5. Wide Receiver (WR) Analysis

Wide receivers are the primary targets in the modern passing game. With 1,617 unique receivers in our dataset, we have the richest data for analysis. Key focus areas: target share, advanced efficiency metrics, situational performance, and the impact of QB context.

---

In [ ]:
# 5.1 — WR Receiving Yards Distribution & Top Players
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution
ax = axes[0]
ax.hist(wr_starters['receiving_yards'].dropna(), bins=40, color=COLORS['WR'], alpha=0.7, edgecolor='white')
ax.axvline(wr_starters['receiving_yards'].mean(), color='black', ls='--', 
           label=f"Mean: {wr_starters['receiving_yards'].mean():.0f}")
ax.set_title('Distribution of Season Receiving Yards')
ax.set_xlabel('Receiving Yards')
ax.legend()

# Top 15 single seasons
top_wr = wr_starters.nlargest(15, 'receiving_yards')[['receiver_player_name','season','receiving_yards']]
top_wr['Label'] = top_wr['receiver_player_name'] + ' (' + top_wr['season'].astype(str) + ')'
ax = axes[1]
bars = ax.barh(top_wr['Label'][::-1], top_wr['receiving_yards'][::-1],
               color=plt.cm.YlOrRd(np.linspace(0.3, 0.9, 15)))
ax.set_xlabel('Receiving Yards')
ax.set_title('Top 15 WR Seasons', fontweight='bold')

# Yearly trend
ax = axes[2]
yearly_wr = wr_starters.groupby('season')['receiving_yards'].mean()
ax.plot(yearly_wr.index, yearly_wr.values, 'o-', color=COLORS['WR'], lw=2)
ax.set_xlabel('Season')
ax.set_ylabel('Avg Receiving Yards')
ax.set_title('Avg WR Receiving Yards Over Time', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 5.2 — WR Advanced Metrics: EPA & Target Share
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# EPA distribution
ax = axes[0, 0]
ax.hist(wr_starters['epa'].dropna(), bins=40, color=COLORS['WR'], alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', ls='--', lw=2, label='Zero EPA')
ax.axvline(wr_starters['epa'].mean(), color='black', ls='--', label=f"Mean: {wr_starters['epa'].mean():.3f}")
ax.set_title('EPA Distribution (WR)')
ax.set_xlabel('Expected Points Added')
ax.legend()

# Target Share vs Yards
ax = axes[0, 1]
valid_ts = wr_starters.dropna(subset=['target_share', 'receiving_yards'])
ax.scatter(valid_ts['target_share'], valid_ts['receiving_yards'], alpha=0.3, s=15, color=COLORS['WR'])
r, _ = stats.pearsonr(valid_ts['target_share'], valid_ts['receiving_yards'])
ax.set_xlabel('Target Share')
ax.set_ylabel('Receiving Yards')
ax.set_title(f'Target Share vs Receiving Yards (r={r:.3f})', fontweight='bold')

# ADOT vs YAC
ax = axes[1, 0]
valid_ay = wr_starters.dropna(subset=['adot', 'yac_per_reception'])
if len(valid_ay) > 20:
    ax.scatter(valid_ay['adot'], valid_ay['yac_per_reception'], alpha=0.3, s=15, color='purple')
    r, _ = stats.pearsonr(valid_ay['adot'], valid_ay['yac_per_reception'])
    ax.set_xlabel('Average Depth of Target')
    ax.set_ylabel('YAC per Reception')
    ax.set_title(f'ADOT vs YAC/Rec (r={r:.3f}): Deep vs YAC Receivers', fontweight='bold')

# Catch Rate vs Yards/Target
ax = axes[1, 1]
valid_cr = wr_starters.dropna(subset=['catch_rate', 'yards_per_target'])
if len(valid_cr) > 20:
    ax.scatter(valid_cr['catch_rate'], valid_cr['yards_per_target'], alpha=0.3, s=15, color='teal')
    r, _ = stats.pearsonr(valid_cr['catch_rate'], valid_cr['yards_per_target'])
    ax.set_xlabel('Catch Rate')
    ax.set_ylabel('Yards per Target')
    ax.set_title(f'Catch Rate vs Y/Tgt (r={r:.3f})', fontweight='bold')

plt.suptitle('WR Advanced Receiving Metrics', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 5.3 — WR Situational Analysis: Red Zone & Quarter Splits
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Red zone targets vs TDs
ax = axes[0]
valid_rz = wr_starters.dropna(subset=['red_zone_targets', 'tds'])
ax.scatter(valid_rz['red_zone_targets'], valid_rz['tds'], alpha=0.3, s=15, color='red')
r, _ = stats.pearsonr(valid_rz['red_zone_targets'], valid_rz['tds'])
ax.set_xlabel('Red Zone Targets')
ax.set_ylabel('Touchdowns')
ax.set_title(f'Red Zone Targets vs TDs (r={r:.3f})', fontweight='bold')

# Yards by quarter (top 100 WRs)
ax = axes[1]
q_cols = ['yards_Q1', 'yards_Q2', 'yards_Q3', 'yards_Q4']
q_cols_valid = [c for c in q_cols if c in wr_starters.columns]
if q_cols_valid:
    top100 = wr_starters.nlargest(100, 'receiving_yards')
    quarter_means = top100[q_cols_valid].mean()
    ax.bar(['Q1', 'Q2', 'Q3', 'Q4'][:len(q_cols_valid)], quarter_means,
           color=['#2196F3', '#4CAF50', '#FF9800', '#F44336'])
    ax.set_ylabel('Avg Yards')
    ax.set_title('Yards by Quarter (Top 100 WR Seasons)', fontweight='bold')

# Win probability splits
ax = axes[2]
wp_cols = ['yards_wp_<25', 'yards_wp_25_45', 'yards_wp_45_55', 'yards_wp_55_75', 'yards_wp_>75']
wp_cols_valid = [c for c in wp_cols if c in wr_starters.columns]
if wp_cols_valid:
    top100 = wr_starters.nlargest(100, 'receiving_yards')
    wp_means = top100[wp_cols_valid].mean()
    labels = ['<25%', '25-45%', '45-55%', '55-75%', '>75%'][:len(wp_cols_valid)]
    ax.bar(labels, wp_means, color=plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(wp_cols_valid))))
    ax.set_xlabel('Win Probability')
    ax.set_ylabel('Avg Yards')
    ax.set_title('Yards by Win Probability (Top 100 WRs)', fontweight='bold')

plt.tight_layout()
plt.show()
print("Conclusion: Top WRs produce heavily in Q2 and when the game is close (45-55% WP).")

In [ ]:
# 5.4 — WR Correlation Heatmap
wr_corr_cols = ['targets', 'receptions', 'receiving_yards', 'tds', 'catch_rate',
                'epa', 'target_share', 'adot', 'yac', 'air_yards', 'explosive_plays',
                'red_zone_targets', 'success_rate']
wr_corr_cols = [c for c in wr_corr_cols if c in wr_starters.columns]
corr = wr_starters[wr_corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='YlOrRd', vmin=-0.5, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('WR Stats Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 5.5 — WR Weather Impact
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Temperature vs yards
ax = axes[0]
valid_temp = wr_starters.dropna(subset=['temp_f', 'receiving_yards'])
if len(valid_temp) > 50:
    bins_t = pd.cut(valid_temp['temp_f'], bins=8)
    binned = valid_temp.groupby(bins_t, observed=True)['receiving_yards'].mean()
    ax.bar(range(len(binned)), binned.values, color=plt.cm.coolwarm(np.linspace(0,1,len(binned))))
    ax.set_xticks(range(len(binned)))
    ax.set_xticklabels([f"{b.left:.0f}-{b.right:.0f}" for b in binned.index], rotation=45, fontsize=8)
    ax.set_xlabel('Temperature (°F)')
    ax.set_ylabel('Avg Receiving Yards')
    ax.set_title('WR Yards by Temperature', fontweight='bold')

# Wind vs yards
ax = axes[1]
valid_wind = wr_starters.dropna(subset=['wind_mph', 'receiving_yards'])
if len(valid_wind) > 50:
    bins_w = pd.cut(valid_wind['wind_mph'], bins=6)
    binned_w = valid_wind.groupby(bins_w, observed=True)['receiving_yards'].mean()
    ax.bar(range(len(binned_w)), binned_w.values, color='steelblue')
    ax.set_xticks(range(len(binned_w)))
    ax.set_xticklabels([f"{b.left:.0f}-{b.right:.0f}" for b in binned_w.index], rotation=45, fontsize=8)
    ax.set_xlabel('Wind (mph)')
    ax.set_ylabel('Avg Receiving Yards')
    ax.set_title('WR Yards by Wind Speed', fontweight='bold')

# Dome vs outdoor
ax = axes[2]
valid_dome = wr_starters.dropna(subset=['is_dome', 'receiving_yards'])
if len(valid_dome) > 50:
    dome_means = valid_dome.groupby('is_dome')['receiving_yards'].mean()
    ax.bar(['Outdoor', 'Dome'], [dome_means.get(0, dome_means.iloc[0]) if 0 in dome_means.index else dome_means.iloc[0],
                                   dome_means.get(1, dome_means.iloc[-1]) if 1 in dome_means.index else dome_means.iloc[-1]],
           color=['steelblue', 'coral'])
    ax.set_ylabel('Avg Receiving Yards')
    ax.set_title('Dome vs Outdoor', fontweight='bold')

plt.tight_layout()
plt.show()
print("Weather impacts receiving production — moderate temperature and low wind are optimal.")

In [ ]:
# 5.6 — QB Context: How QB quality affects WR production
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
valid_qb_ctx = wr_starters.dropna(subset=['qb_cpoe', 'receiving_yards'])
if len(valid_qb_ctx) > 30:
    ax.scatter(valid_qb_ctx['qb_cpoe'], valid_qb_ctx['receiving_yards'], alpha=0.2, s=10, color=COLORS['WR'])
    r, _ = stats.pearsonr(valid_qb_ctx['qb_cpoe'], valid_qb_ctx['receiving_yards'])
    ax.set_xlabel('QB CPOE (Completion % Over Expected)')
    ax.set_ylabel('WR Receiving Yards')
    ax.set_title(f'QB CPOE vs WR Yards (r={r:.3f})', fontweight='bold')

ax = axes[1]
valid_qb_att = wr_starters.dropna(subset=['team_pass_attempts', 'targets'])
if len(valid_qb_att) > 30:
    ax.scatter(valid_qb_att['team_pass_attempts'], valid_qb_att['targets'], alpha=0.2, s=10, color='orange')
    r, _ = stats.pearsonr(valid_qb_att['team_pass_attempts'], valid_qb_att['targets'])
    ax.set_xlabel('Team Pass Attempts')
    ax.set_ylabel('WR Targets')
    ax.set_title(f'Team Pass Volume vs WR Targets (r={r:.3f})', fontweight='bold')

plt.tight_layout()
plt.show()
print("WR production is partly dependent on QB quality and team passing volume.")

---
# 6. Tight End (TE) Analysis

Tight ends occupy a unique hybrid role — part receiver, part blocker. Elite pass-catching TEs like Travis Kelce have transformed the position. Our dataset (27 TEs, 2013–2025) captures this modern era.

---

In [ ]:
# 6.1 — TE Receiving Yards Distribution & Top Players
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution
ax = axes[0]
ax.hist(te_starters['Rec_Yds'].dropna(), bins=25, color=COLORS['TE'], alpha=0.7, edgecolor='white')
ax.axvline(te_starters['Rec_Yds'].mean(), color='black', ls='--',
           label=f"Mean: {te_starters['Rec_Yds'].mean():.0f}")
ax.set_title('Distribution of TE Season Receiving Yards')
ax.set_xlabel('Receiving Yards')
ax.legend()

# Top career TEs
te_career = te.groupby('Player').agg({'Rec_Yds': 'sum', 'Rec_TD': 'sum', 'Rec': 'sum'}).reset_index()
top10_te = te_career.nlargest(10, 'Rec_Yds')
ax = axes[1]
bars = ax.barh(top10_te['Player'][::-1], top10_te['Rec_Yds'][::-1],
               color=plt.cm.Greens(np.linspace(0.3, 0.9, 10)))
ax.set_xlabel('Career Receiving Yards')
ax.set_title('Top 10 TEs by Career Receiving Yards', fontweight='bold')

# Targets over time
ax = axes[2]
yearly_te = te_starters.groupby('Season')['Tgt'].mean()
ax.plot(yearly_te.index, yearly_te.values, 'o-', color=COLORS['TE'], lw=2)
ax.set_xlabel('Season')
ax.set_ylabel('Avg Targets')
ax.set_title('TE Avg Targets per Season Over Time', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 6.2 — TE Efficiency: Catch Rate, Y/R, ADOT
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Catch rate distribution
ax = axes[0]
te_cp = te_starters['catch_pct'].dropna()
if len(te_cp) > 5:
    ax.hist(te_cp, bins=20, color=COLORS['TE'], alpha=0.7, edgecolor='white')
    ax.axvline(te_cp.mean(), color='black', ls='--', label=f'Mean: {te_cp.mean():.1f}%')
    ax.set_title('TE Catch Rate Distribution')
    ax.set_xlabel('Catch %')
    ax.legend()

# Y/R vs Rec
ax = axes[1]
valid_yr = te_starters.dropna(subset=['Rec_Y/R', 'Rec'])
ax.scatter(valid_yr['Rec'], valid_yr['Rec_Y/R'], alpha=0.5, s=30, color=COLORS['TE'])
ax.set_xlabel('Receptions')
ax.set_ylabel('Yards per Reception')
ax.set_title('Receptions vs Y/R', fontweight='bold')

# ADOT vs YAC
ax = axes[2]
valid_te_adv = te_starters.dropna(subset=['adv_ADOT', 'adv_Rec_YAC'])
if len(valid_te_adv) > 5:
    ax.scatter(valid_te_adv['adv_ADOT'], valid_te_adv['adv_Rec_YAC'], alpha=0.5, s=30, color='darkgreen')
    ax.set_xlabel('Average Depth of Target')
    ax.set_ylabel('Yards After Catch')
    ax.set_title('TE ADOT vs YAC', fontweight='bold')

plt.tight_layout()
plt.show()
print("Elite TEs combine high catch rates with solid ADOT — reliable targets across the middle of the field.")

In [ ]:
# 6.3 — TE Correlation Heatmap
te_corr_cols = ['Tgt', 'Rec', 'Rec_Yds', 'Rec_TD', 'Rec_Y/R', 'catch_pct',
                'adv_ADOT', 'adv_Rec_YAC', 'adv_Rec_BrkTkl', 'adv_rec_pass_rating']
te_corr_cols = [c for c in te_corr_cols if c in te_starters.columns]
corr = te_starters[te_corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Greens', vmin=-0.5, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('TE Stats Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 6.4 — TE Player Profiles: Radar Chart
from matplotlib.patches import FancyBboxPatch

# Compare top TEs across key metrics
top_tes = ['Travis Kelce', 'George Kittle', 'Mark Andrews', 'T.J. Hockenson']
metrics = ['Rec', 'Rec_Yds', 'Rec_TD', 'Tgt', 'catch_pct']
metrics = [m for m in metrics if m in te.columns]

fig, axes = plt.subplots(1, len(top_tes), figsize=(5*len(top_tes), 5))
if len(top_tes) == 1:
    axes = [axes]

for ax, te_name in zip(axes, top_tes):
    player_data = te[te['Player'] == te_name]
    if len(player_data) > 0:
        means = player_data[metrics].mean()
        # Normalize to 0-1 for comparison
        maxes = te_starters[metrics].max()
        normalized = (means / maxes * 100).fillna(0)
        
        colors_list = plt.cm.Greens(np.linspace(0.4, 0.8, len(metrics)))
        ax.barh(metrics, normalized, color=colors_list)
        ax.set_xlim(0, 110)
        ax.set_title(te_name, fontweight='bold')
        ax.set_xlabel('% of Max')
    else:
        ax.text(0.5, 0.5, f'{te_name}\nNot Found', ha='center', va='center', transform=ax.transAxes)

plt.suptitle('TE Player Profiles (% of Dataset Maximum)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
# 7. Cross-Position Comparisons

How do the positions compare across common metrics? Let's look at scoring output, age curves, and the evolution of each position.

---

In [ ]:
# 7.1 — Touchdowns by Position
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# TD distributions
ax = axes[0]
td_data = [
    qb_starters['TD'].dropna(),
    rb_starters['Rush_TD'].dropna(),
    wr_starters['tds'].dropna(),
    te_starters['Rec_TD'].dropna()
]
td_labels = ['QB (Pass TD)', 'RB (Rush TD)', 'WR (Rec TD)', 'TE (Rec TD)']
bp = ax.boxplot(td_data, labels=td_labels, patch_artist=True)
for patch, color in zip(bp['boxes'], [COLORS['QB'], COLORS['RB'], COLORS['WR'], COLORS['TE']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Touchdowns')
ax.set_title('Season TD Distribution by Position', fontweight='bold')

# Average TDs over time
ax = axes[1]
qb_yr = qb_starters.groupby('Season')['TD'].mean()
rb_yr = rb_starters.groupby('Season')['Rush_TD'].mean()
wr_yr = wr_starters.groupby('season')['tds'].mean()
te_yr = te_starters.groupby('Season')['Rec_TD'].mean()

ax.plot(qb_yr.index, qb_yr.values, 'o-', color=COLORS['QB'], label='QB Pass TD', lw=2)
ax.plot(rb_yr.index, rb_yr.values, 's-', color=COLORS['RB'], label='RB Rush TD', lw=2)
ax.plot(wr_yr.index, wr_yr.values, '^-', color=COLORS['WR'], label='WR Rec TD', lw=2)
ax.plot(te_yr.index, te_yr.values, 'd-', color=COLORS['TE'], label='TE Rec TD', lw=2)
ax.set_xlabel('Season')
ax.set_ylabel('Avg TDs')
ax.set_title('Average Season TDs by Position Over Time', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 7.2 — Age Curves Across Positions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Primary production by age
ax = axes[0]
qb_age = qb_starters.groupby('Age')['Yds'].mean()
rb_age = rb_starters.groupby('Age')['Rush_Yds'].mean()
te_age = te_starters.groupby('Age')['Rec_Yds'].mean()

valid_ages = range(21, 40)
for data, label, color, ylabel in [
    (qb_age, 'QB Pass Yds', COLORS['QB'], 'Yards'),
    (rb_age, 'RB Rush Yds', COLORS['RB'], 'Yards'),
    (te_age, 'TE Rec Yds', COLORS['TE'], 'Yards'),
]:
    ages = [a for a in valid_ages if a in data.index]
    vals = [data[a] for a in ages]
    ax.plot(ages, vals, 'o-', label=label, color=color, lw=2)

ax.set_xlabel('Age')
ax.set_ylabel('Avg Yards (Primary Stat)')
ax.set_title('Age Curves by Position', fontweight='bold')
ax.legend()

# Games started by age
ax = axes[1]
for df, label, color, age_col in [
    (qb_starters, 'QB', COLORS['QB'], 'Age'),
    (rb_starters, 'RB', COLORS['RB'], 'Age'),
    (te_starters, 'TE', COLORS['TE'], 'Age'),
]:
    age_gs = df.groupby(age_col)['GS'].mean()
    ages = [a for a in valid_ages if a in age_gs.index]
    vals = [age_gs[a] for a in ages]
    ax.plot(ages, vals, 'o-', label=label, color=color, lw=2)

ax.set_xlabel('Age')
ax.set_ylabel('Avg Games Started')
ax.set_title('Games Started by Age & Position', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()
print("QBs peak later and last longer. RBs peak early (25-27) and decline fast. TEs fall in between.")

In [ ]:
# 7.3 — Dataset Size Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rows per position
sizes = {'QB': len(qb), 'RB': len(rb), 'TE': len(te), 'WR': len(wr)}
ax = axes[0]
bars = ax.bar(sizes.keys(), sizes.values(), color=[COLORS[p] for p in sizes.keys()])
for bar, val in zip(bars, sizes.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:,}',
            ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Number of Player-Seasons')
ax.set_title('Dataset Size by Position', fontweight='bold')

# Unique players
players = {
    'QB': qb['Player'].nunique(),
    'RB': rb['Player'].nunique(),
    'TE': te['Player'].nunique(),
    'WR': wr['receiver_player_name'].nunique()
}
ax = axes[1]
bars = ax.bar(players.keys(), players.values(), color=[COLORS[p] for p in players.keys()])
for bar, val in zip(bars, players.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:,}',
            ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Unique Players')
ax.set_title('Unique Players by Position', fontweight='bold')

plt.tight_layout()
plt.show()

---
# 8. Feature Importance Preview

Before building models in Notebook 3, let's identify which features are most correlated with our target variables. This helps us understand what stats best predict performance.

---

In [ ]:
# 8.1 — Top Correlations with QB Passing Yards
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# QB: correlations with Yds
ax = axes[0, 0]
qb_num = qb_starters.select_dtypes(include=[np.number])
corr_yds = qb_num.corr()['Yds'].drop('Yds').sort_values(ascending=False)
top_pos = corr_yds.head(15)
top_neg = corr_yds.tail(5)
combined = pd.concat([top_pos, top_neg])
colors_c = ['green' if v > 0 else 'red' for v in combined]
combined.plot.barh(ax=ax, color=colors_c)
ax.set_title('Top Correlations with QB Passing Yards', fontweight='bold')
ax.set_xlabel('Pearson r')

# RB: correlations with Rush_Yds
ax = axes[0, 1]
rb_num = rb_starters.select_dtypes(include=[np.number])
if 'Rush_Yds' in rb_num.columns:
    corr_rush = rb_num.corr()['Rush_Yds'].drop('Rush_Yds').sort_values(ascending=False)
    top_r = pd.concat([corr_rush.head(15), corr_rush.tail(5)])
    colors_r = ['green' if v > 0 else 'red' for v in top_r]
    top_r.plot.barh(ax=ax, color=colors_r)
    ax.set_title('Top Correlations with RB Rushing Yards', fontweight='bold')
    ax.set_xlabel('Pearson r')

# WR: correlations with receiving_yards
ax = axes[1, 0]
wr_num = wr_starters.select_dtypes(include=[np.number])
if 'receiving_yards' in wr_num.columns:
    corr_rec = wr_num.corr()['receiving_yards'].drop('receiving_yards').sort_values(ascending=False)
    top_w = pd.concat([corr_rec.head(15), corr_rec.tail(5)])
    colors_w = ['green' if v > 0 else 'red' for v in top_w]
    top_w.plot.barh(ax=ax, color=colors_w)
    ax.set_title('Top Correlations with WR Receiving Yards', fontweight='bold')
    ax.set_xlabel('Pearson r')

# TE: correlations with Rec_Yds
ax = axes[1, 1]
te_num = te_starters.select_dtypes(include=[np.number])
if 'Rec_Yds' in te_num.columns:
    corr_te = te_num.corr()['Rec_Yds'].drop('Rec_Yds').sort_values(ascending=False)
    top_t = pd.concat([corr_te.head(15), corr_te.tail(5)])
    colors_t = ['green' if v > 0 else 'red' for v in top_t]
    top_t.plot.barh(ax=ax, color=colors_t)
    ax.set_title('Top Correlations with TE Receiving Yards', fontweight='bold')
    ax.set_xlabel('Pearson r')

plt.suptitle('Feature-Target Correlations by Position', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("These correlations guide feature selection for our ML models in Notebook 3.")

In [ ]:
# 8.2 — Pairplot of top QB features
top_qb_features = ['Yds', 'TD', 'Cmp%', 'Rate', 'Y/A', 'Win%']
top_qb_features = [c for c in top_qb_features if c in qb_starters.columns]
sample = qb_starters[top_qb_features].dropna().sample(min(200, len(qb_starters)), random_state=42)

g = sns.pairplot(sample, diag_kind='kde', plot_kws={'alpha': 0.3, 's': 15, 'color': COLORS['QB']},
                 diag_kws={'color': COLORS['QB']})
g.figure.suptitle('QB Key Features Pairplot', y=1.02, fontsize=14, fontweight='bold')
plt.show()

---
# 9. Key Conclusions & Insights

### Quarterback Findings
1. **Passing revolution** — Yards, TDs, completion %, and passer rating all show strong upward trends over the decades
2. **Win predictors** — Passer Rating, Y/A, and Completion % correlate most strongly with winning
3. **Turnovers matter** — INT% shows a clear negative correlation with Win%
4. **Stat padding is real** — QBs on losing teams accumulate yards in garbage time
5. **Dual-threat evolution** — Modern QBs (Jackson, Allen) combine passing with significant rushing

### Running Back Findings
1. **Declining workhorse** — Total carries per back have decreased as RBBC (running back by committee) has become prevalent
2. **Pass-catching evolution** — The receiving share of RB production has steadily increased
3. **Short career window** — RBs peak at 25–27 and decline rapidly, unlike QBs who can play into their late 30s
4. **Broken tackles predict success** — Advanced metric `adv_Rush_BrkTkl` correlates strongly with total yardage

### Wide Receiver Findings
1. **Target share is king** — The strongest predictor of WR production is target share
2. **ADOT vs YAC tradeoff** — Deep threats and YAC monsters represent different receiver archetypes
3. **Weather matters** — Wind and extreme temperatures negatively impact receiving production
4. **QB dependency** — WR success is partly a function of QB quality (CPOE) and team pass volume

### Tight End Findings
1. **Position transformation** — Modern TEs are increasingly used as primary receivers
2. **Travis Kelce effect** — The elite TEs are statistical outliers, far above the positional average
3. **Catch rate reliability** — TEs are among the most reliable targets with high catch rates

### Cross-Position Insights
1. **QBs age best** — Quarterbacks maintain production much longer than other positions
2. **WR dataset richest** — With 1,617 unique players, the WR dataset offers the most analytical depth
3. **Feature selection** — Different statistical predictors matter for each position, requiring position-specific models

---
*Next: Notebook 3 — Models & Results → Building predictive models for each position*